# Week 4 runs

Runs the pipeline end to end on the current split and prints the reports it writes. Nothing is computed here that the scripts do not compute, so the notebook and the files in `results/` cannot disagree.

Cells follow three rules. One purpose per cell, with a caption that states what it reports. Numbers are printed, not typed into prose. Anything that has to be reproducible lives in `src/`, the notebook calls it.

In [1]:
import os
import subprocess
import sys

root = '/kaggle/working/repo'
if not os.path.isdir(os.path.join(root, 'src')):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/ByteBandit0608/reverse-dictionary-capstone', root])
else:
    subprocess.run(['git', '-C', root, 'pull', '-q', '--ff-only', 'origin', 'main'])
os.chdir(root)
print(open(os.path.join(root, '.git', 'HEAD')).read().strip())
sys.version

ref: refs/heads/main


'3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]'

## 1. Split and metrics

`src/run_all.py` runs the split, both lexical baselines, the feature report and the failure analysis in that order.

In [2]:
subprocess.run([sys.executable, 'src/run_all.py'])


=== make_splits.py ===
wrote splits
rows_raw                         176009
rows                             135907
rows_dropped_bad_def             215
rows_dropped_named               207
rows_named_in_word               207
rows_named_in_definition         0
rows_named_word_exact            207
rows_dropped_empty_word          4
rows_dropped_crossref            5616
rows_dropped_dupes               232
rows_dropped_self_ref            34039
pct_rows_self_ref                19.45
pct_rows_self_ref_exact          13.33
n_words                          95286
n_words_train                    76229
n_words_val                      9529
n_words_test                     9528
rows_train                       108759
rows_val                         13395
rows_test                        13753
n_queries                        500
pct_multi_sense                  20.37
pct_shared_def                   7.86
max_tie                          9
ceiling_strict_top1              0.995
pct_reachable

CompletedProcess(args=['/usr/bin/python3', 'src/run_all.py'], returncode=0)

## 2. Pretrained embeddings

No fine tuning, one model per run. `src/run_dense.py` writes `results/embeddings.md` and merges into the json, so a model added later lands in the same table. When the third name is posted it runs the same way.

In [3]:
subprocess.run([sys.executable, 'src/run_dense.py',
                    '--models', 'sentence-transformers/all-mpnet-base-v2'])

corpus rows=135907 words=95286 queries=500  models=sentence-transformers/all-mpnet-base-v2

[sentence-transformers/all-mpnet-base-v2]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2354.00it/s]


      using cached vectors (135907, 768)
  all_mpnet_base_v2            in_dictionary  R@1=0.992 R@10=1.000 R@100=1.000  medrank=1  self@1=99.2%
  all_mpnet_base_v2            leave_one_out  R@1=0.050 R@10=0.170 R@100=0.394  medrank=319  self@1=0.0%

wrote results/embeddings.md and .json


CompletedProcess(args=['/usr/bin/python3', 'src/run_dense.py', '--models', 'sentence-transformers/all-mpnet-base-v2'], returncode=0)

## 3. The reports

Printed as they were written, so this notebook is a readable copy of the run.

In [4]:
for name in ('baselines', 'embeddings', 'minilm', 'mpnet', 'pos_roles', 'error_analysis'):
    p = os.path.join(root, 'results', name + '.md')
    if os.path.exists(p):
        print('\n' + '=' * 72 + '\n' + name + '\n' + '=' * 72)
        print(open(p).read())


baselines
# baselines, bm25 and tfidf

from src/run_baselines.py. cleaned corpus is 135,907 rows over 95,286 headwords, 500 fixed queries from the test split, seed 2026. took 30.8s.

| method | protocol | R@1 | R@10 | R@100 | medrank | R@1 tol | own row at rank 1 |
|---|---|---|---|---|---|---|---|
| bm25 | in_dictionary | 0.990 | 1.000 | 1.000 | 1 | 0.996 | 99.0% |
| bm25 | leave_one_out | 0.046 | 0.152 | 0.322 | 945 | 0.056 | 0.0% |
| tfidf | in_dictionary | 0.992 | 1.000 | 1.000 | 1 | 0.998 | 99.2% |
| tfidf | leave_one_out | 0.048 | 0.162 | 0.350 | 884 | 0.058 | 0.0% |
| word_freq | in_dictionary | 0.000 | 0.000 | 0.000 | 1000 | 0.000 | - |
| random | in_dictionary | 0.000 | 0.002 | 0.004 | 1000 | 0.000 | - |

notes

- in_dictionary cant be quoted on its own. the query string is sitting in the index, so a lexical method gets it by matching text. that is lookup, not understanding the definition.
- leave_one_out takes that row out. 0.0% of these queries have no other row for their w

## 4. One comparison across methods

Reads the result files instead of restating them, and flags any file produced against an earlier version of the corpus so rows from two query sets never sit in one table unread. Leave-one-out is the column to compare on, in-dictionary is kept because the self-match is what is controlled for.

In [5]:
import json
import pandas as pd

man = pd.read_json(os.path.join(root, 'splits', 'manifest.json'), typ='series')
rows = []

def add(name, r, meta):
    note = ''
    if meta and meta.get('rows') != man.get('rows'):
        note = ' (earlier corpus: %s rows there, %s here)' % (meta.get('rows'), man.get('rows'))
    rows.append(dict(method=str(r.get('method') or name) + note, protocol=r['protocol'],
                     r1=r['recall@1'], r10=r['recall@10'], r100=r['recall@100'],
                     medrank=r['median_rank']))

for name in ('baselines', 'minilm', 'mpnet', 'embeddings'):
    p = os.path.join(root, 'results', name + '.json')
    if not os.path.exists(p):
        continue
    d = json.load(open(p))
    for r in d.get('table', []):
        add(name, r, d.get('meta'))

t = pd.DataFrame(rows)
if t.duplicated(['method', 'protocol']).any():
    print('note: a model is in more than one result file, keeping the current corpus row')
    print('      two scripts should not cover the same model, pick one')
t['fresh'] = ~t['method'].str.contains('earlier corpus')
t = (t.sort_values(['method', 'protocol', 'fresh'], ascending=[True, True, False])
       .drop_duplicates(['method', 'protocol'], keep='first')
       .drop(columns='fresh'))
for proto in ('leave_one_out', 'in_dictionary'):
    print(proto)
    print(t[t.protocol == proto].sort_values('r100', ascending=False).to_string(index=False))
    print()

leave_one_out
                                                 method      protocol    r1   r10  r100  medrank
                                      all_mpnet_base_v2 leave_one_out 0.050 0.170 0.394      319
                                                  tfidf leave_one_out 0.048 0.162 0.350      884
                                                   bm25 leave_one_out 0.046 0.152 0.322      945
minilm (earlier corpus: 170140 rows there, 135907 here) leave_one_out 0.028 0.132 0.284     1000

in_dictionary
                                                 method      protocol    r1   r10  r100  medrank
                                      all_mpnet_base_v2 in_dictionary 0.992 1.000 1.000        1
                                                   bm25 in_dictionary 0.990 1.000 1.000        1
minilm (earlier corpus: 170140 rows there, 135907 here) in_dictionary 0.998 1.000 1.000        1
                                                  tfidf in_dictionary 0.992 1.000 1.000        1
 